# Feature 2 — HR Document Search and Retrieval

**SmartStart AI | AAI-530**

Run the cells in order. No Git installation or LLM API key required. The notebook finds seven TXT documents locally, in a ZIP archive, or downloads them from your GitHub feature branch. Retrieval uses TF-IDF and cosine similarity and returns source metadata.

## 1. Install dependency

In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\aisaac\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


## 2. Imports and configuration

In [2]:
from pathlib import Path
from dataclasses import dataclass, replace
import io
import re
import zipfile
import urllib.request
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

REPO = "millerr2009-AAI/aai-530-hr-rag-assistant"
BRANCH = "feature/HR-Document-Search-and-Retrieval"
EXPECTED_FILES = [
    "01_First_Week_Onboarding_Tasks.txt",
    "02_Required_HR_Documents.txt",
    "03_Benefits_Enrollment.txt",
    "04_Employee_Training.txt",
    "05_Equipment_and_Account_Requests.txt",
    "06_HR_Policies.txt",
    "07_HR_Knowledge_Base.txt",
]
print("Working folder:", Path.cwd())

Working folder: C:\Users\aisaac


## 3. Locate or download the seven HR documents

First checks local folders, then ZIP files in the notebook folder, then downloads the GitHub branch ZIP with Python (no `git` executable needed). If GitHub is private or offline, place the supplied HR ZIP beside this notebook and rerun.

In [3]:
def locate_documents(base="."):
    base = Path(base).resolve()
    candidate_dirs = [base, base / "hr_documents", base / "HR_RAG_Document_Collection"]
    for directory in candidate_dirs:
        if all((directory / f).is_file() for f in EXPECTED_FILES):
            print("Using local HR documents:", directory)
            return directory

    # Find the files in a nested directory if already extracted.
    for first in base.rglob(EXPECTED_FILES[0]):
        directory = first.parent
        if all((directory / f).is_file() for f in EXPECTED_FILES):
            print("Using extracted HR documents:", directory)
            return directory

    output = base / "hr_documents"
    output.mkdir(exist_ok=True)
    archives = sorted(base.glob("*.zip"))
    if not archives:
        url = f"https://codeload.github.com/{REPO}/zip/refs/heads/{BRANCH}"
        print("Downloading GitHub feature branch archive (no Git required)...")
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                archive_data = response.read()
            archive = zipfile.ZipFile(io.BytesIO(archive_data))
        except Exception as exc:
            raise RuntimeError(
                "Could not download the GitHub branch. If the repository is private or "
                "you are offline, put HR_RAG_Document_Team8.zip next to the notebook."
            ) from exc
    else:
        print("Using local ZIP:", archives[0].name)
        archive = zipfile.ZipFile(archives[0])

    with archive:
        for filename in EXPECTED_FILES:
            matches = [name for name in archive.namelist()
                       if Path(name).name == filename and not name.endswith("/")]
            if not matches:
                raise FileNotFoundError(f"Missing from archive: {filename}")
            (output / filename).write_bytes(archive.read(matches[0]))
    print("Extracted", len(EXPECTED_FILES), "HR TXT documents into", output)
    return output

document_folder = locate_documents()
print("Documents:")
for filename in EXPECTED_FILES:
    print(" -", filename)

Using extracted HR documents: C:\Users\aisaac\Downloads\HR_RAG_Document_Team8\HR_RAG_Document_Collection
Documents:
 - 01_First_Week_Onboarding_Tasks.txt
 - 02_Required_HR_Documents.txt
 - 03_Benefits_Enrollment.txt
 - 04_Employee_Training.txt
 - 05_Equipment_and_Account_Requests.txt
 - 06_HR_Policies.txt
 - 07_HR_Knowledge_Base.txt


## 4. Preserve metadata and split documents by section

In [4]:
@dataclass
class Passage:
    document_id: str
    title: str
    section: str
    text: str
    source: str
    score: float = 0.0
    version: str = ""


def parse_file(path: Path, chunk_words: int = 150, overlap: int = 30) -> list[Passage]:
    if chunk_words <= 0 or not (0 <= overlap < chunk_words):
        raise ValueError("Require chunk_words > overlap >= 0")
    raw = path.read_text(encoding="utf-8-sig")
    title_match = re.search(r"(?m)^#\s+(.+)$", raw)
    id_match = re.search(r"(?mi)^Document ID:\s*(.+)$", raw)
    version_match = re.search(r"(?mi)^Version:\s*(.+)$", raw)
    title = title_match.group(1).strip() if title_match else path.stem.split("_", 1)[-1].replace("_", " ")
    document_id = id_match.group(1).strip() if id_match else path.stem
    version = version_match.group(1).strip() if version_match else ""

    # Actual source TXT files use Markdown ## section headings.
    headings = list(re.finditer(r"(?m)^#{2,3}\s+(.+?)\s*$", raw))
    sections = []
    if headings:
        for i, heading in enumerate(headings):
            start = heading.end()
            end = headings[i+1].start() if i+1 < len(headings) else len(raw)
            sections.append((heading.group(1).strip(), raw[start:end].strip()))
    else:
        sections.append(("General", raw.strip()))

    passages = []
    for section, body in sections:
        words = body.split()
        step = chunk_words - overlap
        for start in range(0, len(words), step):
            chunk = words[start:start + chunk_words]
            if chunk:
                passages.append(Passage(document_id, title, section, " ".join(chunk), path.name, version=version))
            if start + chunk_words >= len(words):
                break
    return passages

all_passages = []
for filename in EXPECTED_FILES:
    all_passages.extend(parse_file(document_folder / filename))
assert all_passages, "No searchable passages were created"
print("Searchable passages:", len(all_passages))
print("Example metadata:", all_passages[0].document_id, "|", all_passages[0].title, "|", all_passages[0].section)

Searchable passages: 36
Example metadata: 01_First_Week_Onboarding_Tasks | First-Week Onboarding Tasks | Before day one


## 5. Build the TF-IDF index and implement required `retrieve()`

In [5]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
corpus = [f"{p.title} {p.section} {p.text}" for p in all_passages]
document_matrix = vectorizer.fit_transform(corpus)

def retrieve(question: str, k: int = 4) -> list[Passage]:
    if not isinstance(question, str) or not question.strip() or k <= 0:
        return []
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, document_matrix).ravel()
    ranked = scores.argsort()[::-1]
    return [replace(all_passages[i], score=float(scores[i]))
            for i in ranked if scores[i] > 0][:k]

print("Index ready:", document_matrix.shape[0], "passages")

Index ready: 36 passages


## 6. Test one question and show sources

In [6]:
question = "How do I enroll in benefits?"
results = retrieve(question, k=4)
print("QUESTION:", question)
for rank, passage in enumerate(results, 1):
    print(f"\nRESULT {rank} | score={passage.score:.3f}")
    print("Title:", passage.title)
    print("Document ID:", passage.document_id)
    print("Section:", passage.section)
    print("Source:", passage.source)
    print("Version:", passage.version)
    print("Passage:", passage.text)

QUESTION: How do I enroll in benefits?

RESULT 1 | score=0.236
Title: HR Knowledge Base
Document ID: 07_HR_Knowledge_Base
Section: Frequently asked questions
Source: 07_HR_Knowledge_Base.txt
Version: 1.0
Passage: Q: Where do I submit new-hire forms? A: In the secure HR portal. Q: Where are my training assignments? A: In the LMS. Q: How do I request a laptop? A: Submit a hardware request in the IT service desk. Q: How do I choose benefits? A: Review and enroll through the benefits portal.

RESULT 2 | score=0.133
Title: Benefits Enrollment
Document ID: 03_Benefits_Enrollment
Section: How to enroll
Source: 03_Benefits_Enrollment.txt
Version: 1.0
Passage: Open the secure benefits portal, review plan summaries and dependent eligibility, compare coverage and payroll deductions, select plans, add eligible dependents, and save the confirmation receipt.

RESULT 3 | score=0.082
Title: Equipment and Account Requests
Document ID: 05_Equipment_and_Account_Requests
Section: First login and MFA
Sourc

## 7. Independent retrieval tests

The checks below verify that the expected document appears in the top four results for each category. They are separate from the interactive question interface.

In [7]:
test_cases = [
    ("What should I do during my first week?", "01_First_Week_Onboarding_Tasks.txt"),
    ("What HR documents do I need to submit?", "02_Required_HR_Documents.txt"),
    ("How do I enroll in benefits?", "03_Benefits_Enrollment.txt"),
    ("Where do I find required employee training?", "04_Employee_Training.txt"),
    ("How do I request a laptop or software access?", "05_Equipment_and_Account_Requests.txt"),
    ("How do I report workplace harassment?", "06_HR_Policies.txt"),
    ("Who should I contact about payroll?", "07_HR_Knowledge_Base.txt"),
]
passed = 0
for question, expected in test_cases:
    found = retrieve(question, k=4)
    ok = any(p.source == expected for p in found)
    passed += ok
    print(("PASS" if ok else "REVIEW"), "|", question)
    print("  Expected:", expected)
    print("  Returned:", [p.source for p in found])
print(f"\nExpected document in top 4: {passed}/{len(test_cases)}")
assert len(all_passages) > 0
assert retrieve("", k=4) == []
assert retrieve("benefits", k=0) == []
for q, _ in test_cases:
    for p in retrieve(q, k=4):
        assert p.document_id and p.title and p.section and p.source and p.text
        assert 0 < p.score <= 1
print("Metadata and retrieval contract checks passed. Review relevance above.")

PASS | What should I do during my first week?
  Expected: 01_First_Week_Onboarding_Tasks.txt
  Returned: ['01_First_Week_Onboarding_Tasks.txt', '01_First_Week_Onboarding_Tasks.txt', '01_First_Week_Onboarding_Tasks.txt', '01_First_Week_Onboarding_Tasks.txt']
PASS | What HR documents do I need to submit?
  Expected: 02_Required_HR_Documents.txt
  Returned: ['07_HR_Knowledge_Base.txt', '02_Required_HR_Documents.txt', '02_Required_HR_Documents.txt', '02_Required_HR_Documents.txt']
PASS | How do I enroll in benefits?
  Expected: 03_Benefits_Enrollment.txt
  Returned: ['07_HR_Knowledge_Base.txt', '03_Benefits_Enrollment.txt', '05_Equipment_and_Account_Requests.txt', '03_Benefits_Enrollment.txt']
PASS | Where do I find required employee training?
  Expected: 04_Employee_Training.txt
  Returned: ['04_Employee_Training.txt', '04_Employee_Training.txt', '04_Employee_Training.txt', '04_Employee_Training.txt']
PASS | How do I request a laptop or software access?
  Expected: 05_Equipment_and_Accoun

## 8. Interactive HR question prompt

Run this cell to ask multiple questions. Type **exit** to stop. This returns document passages rather than LLM-generated answers.

In [ ]:
print("SMARTSTART AI — HR DOCUMENT SEARCH")
print("Ask an HR question. Type 'exit' to finish.")
while True:
    question = input("\nAsk an HR question: ").strip()
    if question.lower() in ("exit", "quit"):
        print("Search session ended.")
        break
    if not question:
        print("Please enter a question.")
        continue
    results = retrieve(question, k=4)
    if not results:
        print("No matching passages found.")
        continue
    for rank, passage in enumerate(results, 1):
        print(f"\n[{rank}] {passage.title} — {passage.section} (score {passage.score:.3f})")
        print("Source:", passage.source, "| Document ID:", passage.document_id)
        print(passage.text)
        print("-" * 60)

SMARTSTART AI — HR DOCUMENT SEARCH
Ask an HR question. Type 'exit' to finish.


## 9. Integration note

For the HR chatbot, reuse the `Passage` class and `retrieve(question, k=4)` function. This notebook tests retrieval independently; the chatbot can later pass returned passages into its LLM context.

**GitHub branch:** `feature/HR-Document-Search-and-Retrieval`